<a href="https://colab.research.google.com/github/Filza-coder/Filza-coder.github.io/blob/main/Copy_of_01_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01 — Data Preparation
**Study:** GeoAI-Based Spatial Prediction of Airborne Pollen and Fungal Spore Concentrations  
**Author:** Fatima Filza Hassan | IGIS, NUST Islamabad  
**Dataset:** NUST H-12 campus aerobiological survey, April 2016

## What this notebook does
1. Loads raw pollen/fungus concentration data from `complete data.xlsx`  
2. Loads species-level fungal counts from `species data.xlsx`  
3. Converts raw species counts → concentration (grain/m³) using the **Rotorod formula**  
4. Engineers wind direction circular features: `sin(WD)` and `cos(WD)`  
5. Computes LULC features per station from shapefiles (200 m buffer)  
6. Saves two clean CSV files used by all subsequent notebooks:
   - `df_analysis.csv` — 64 rows (6 stations × 12 days)
   - `df_lulc.csv`     — 6 rows (one per station)

## Run on Google Colab
```python
from google.colab import drive
drive.mount('/content/drive')
```
Then update `DATA_DIR` below to point to your Google Drive folder.

In [ ]:
import os
import sys

# Detect if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone repo if not already cloned
    if not os.path.exists('/content/geoai-bioaerosol-prediction'):
        print('📥 Cloning repository...')
        !git clone https://github.com/Filza-coder/geoai-bioaerosol-prediction /content/geoai-bioaerosol-prediction -q

    # Change to repo directory
    os.chdir('/content/geoai-bioaerosol-prediction')
    print(f'✅ Working directory: {os.getcwd()}')

    # Install dependencies
    print('📦 Installing dependencies...')
    !pip install openpyxl geopandas shapely pyproj scikit-learn shap seaborn pandas -q
else:
    print('✅ Running locally (not in Colab)')

# Verify data exists
if not os.path.exists('complete data.xlsx'):
    print('❌ ERROR: Data file not found!')
    print(f'Current dir: {os.getcwd()}')
    print(f'Files: {os.listdir(".")[:10]}')
else:
    print('✅ Data file found!')

✅ Working directory: /content/geoai-bioaerosol-prediction
📦 Installing dependencies...
✅ Data file found!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ═══════════════════════════════════════════════════════════════
# COLAB SETUP — run this cell first if using Google Colab
# ═══════════════════════════════════════════════════════════════
import os, sys

# Option A: Clone the GitHub repo directly in Colab (recommended)
# !git clone https://github.com/Filza-coder/geoai-bioaerosol-prediction.git
# os.chdir('geoai-bioaerosol-prediction')

# Option B: Mount Google Drive and navigate to your folder
# from google.colab import drive
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/geoai-bioaerosol-prediction')

# Install dependencies
# !pip install openpyxl geopandas shapely pyproj scikit-learn shap seaborn -q

print('Current directory:', os.getcwd())
print('Python:', sys.version[:10])

Current directory: /content/geoai-bioaerosol-prediction
Python: 3.12.13 (m


In [ ]:
# ── Install dependencies (uncomment on Colab) ─────────────────────
# !pip install openpyxl geopandas shapely pyproj -q

In [ ]:
import openpyxl, re, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer
warnings.filterwarnings('ignore')

# ── File paths — update for your environment ────────────────────────
# On Google Colab: '/content/drive/MyDrive/your_folder/data/'
# Corrected: Data files are directly in the current working directory.
DATA_DIR        = ''
COMPLETE_DATA = 'complete data.xlsx'
SPECIES_DATA  = 'species data.xlsx'
SHP_VEGETATION  = 'vegetation.shp'
SHP_GRASSLAND   =  'grassland1.shp'
SHP_BARREN      =  'barren_land.shp'
SHP_BUILDINGS   =  'Buildingsfinal2.shp' # Corrected filename
SHP_TREE_ROAD   =  'alongroadsfinal.shp'
SHP_TREE_GRASS  =  'insidegrasstotal.shp'

print('Libraries loaded successfully.')

Libraries loaded successfully.


## Step 1 — Load meteorological + total pollen/fungus data
Source: `complete data.xlsx → Sheet1`  
Rows 1–4 are headers; data starts at row 5.

**Column mapping (1-based):**
| Col | Variable |
|-----|----------|
| 3 | Station ID (1–6) |
| 5 | Date (as Excel DATE formula string) |
| 6 | `pollen_conc` — total pollen (grain/m³) |
| 7 | `fungus_conc` — total fungus (grain/m³) |
| 8 | GHI (W/m²) |
| 11 | Tamb (°C) |
| 12 | RH (%) |
| 13 | WS (m/s) |
| 15 | WD (degrees N) |
| 17 | BP (hPa) |
| 19 | N_pollen — raw microscopy count |
| 20 | N_fungus — raw microscopy count |

In [ ]:
wb  = openpyxl.load_workbook(os.path.join(DATA_DIR, COMPLETE_DATA.lstrip('/')))
ws  = wb['Sheet1']

met_rows = []
for r in range(5, ws.max_row + 1):
    row = [ws.cell(r, c).value for c in range(1, 22)]
    if row[0] is None:
        continue
    # Parse date from Excel DATE() formula, e.g. '=DATE(2016,4,4)'
    m = re.search(r'DATE\((\d+),(\d+),(\d+)\)', str(row[4]))
    date = f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}" if m else None
    met_rows.append({
        'date':        date,
        'station':     int(row[2]),
        'pollen_conc': row[5],   # grain/m³  (already converted in source file)
        'fungus_conc': row[6],   # grain/m³
        'GHI':         row[7],   # W/m²
        'Tamb':        row[10],  # °C
        'RH':          row[11],  # %
        'WS':          row[12],  # m/s
        'WD':          row[14],  # degrees N
        'BP':          row[16],  # hPa
        'N_pollen':    row[18],  # raw count
        'N_fungus':    row[19],  # raw count
    })

df_met = pd.DataFrame(met_rows)
print(f'Loaded {len(df_met)} rows')
print(f'Stations: {sorted(df_met["station"].unique())}')
print(f'Dates: {sorted(df_met["date"].unique())}')
df_met[['date','station','pollen_conc','fungus_conc','GHI','Tamb','RH','WS','WD']].head(8)

Loaded 64 rows
Stations: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Dates: ['2016-04-04', '2016-04-05', '2016-04-06', '2016-04-07', '2016-04-08', '2016-04-12', '2016-04-13', '2016-04-14', '2016-04-15', '2016-04-18', '2016-04-19', '2016-04-20']


,date,station,pollen_conc,fungus_conc,GHI,Tamb,RH,WS,WD
0,2016-04-04,1,19.22,215.96,814,25.0,50,4.7,270
1,2016-04-04,2,293.16,181.85,814,25.0,50,4.7,270
2,2016-04-04,3,7.81,51.79,814,25.0,50,4.7,270
3,2016-04-04,5,5.53,121.17,814,25.0,50,4.7,270
4,2016-04-05,1,154.34,40.76,726,28.1,33,1.7,124
5,2016-04-05,2,76.08,123.36,726,28.1,33,1.7,124
6,2016-04-05,3,12.50,50.54,726,28.1,33,1.7,124
7,2016-04-05,4,45.65,929.34,726,28.1,33,1.7,124


## Step 2 — Derive sampling volume V per date
### The Rotorod impaction formula (Brown, 1993; Adhikari et al., 2003)

$$V \;(m^3) = \text{Rod\_area} \times D \times \pi \times RPM \times t_{sampled}$$

| Parameter | Value | How determined |
|-----------|-------|----------------|
| Rod area | 0.0015 × 0.035 × 2 = **1.05 × 10⁻⁴ m²** | Thesis Table 3.1 |
| D (rotor head diameter) | **0.1148 m** | Reverse-engineered from N/C pairs |
| RPM | **2700** | Confirmed by tachometer |
| t_sampled | 13–30 min per day | Programmable digital timer |

Because all 6 stations share the same timer on a given day, **V is constant per date**.  
We derive V directly from existing data: **V = N_pollen / pollen_conc** (most reliable approach).

In [ ]:
# Device constants
ROD_W    = 0.0015   # rod width (m)
ROD_H    = 0.035    # rod height (m)
ROD_AREA = ROD_W * ROD_H * 2   # both rods = 1.05e-4 m²
D        = 0.1148   # rotor head diameter (m) — derived from N/C data pairs
RPM      = 2700
PI       = np.pi

print(f'Rod area  = {ROD_W} × {ROD_H} × 2 = {ROD_AREA:.6f} m²')
print(f'D         = {D} m')
print(f'RPM       = {RPM}')
print(f'Formula   : C (grain/m³) = N / V')
print(f'           V = rod_area × D × π × RPM × t_sampled')

# Derive V per date from existing N and C columns
df_met['V'] = df_met['N_pollen'] / df_met['pollen_conc']
date_V = df_met.groupby('date')['V'].median().round(4)

print('\nSampling volume V per date:')
for date, v in date_V.items():
    t_min = v / (ROD_AREA * D * PI * RPM)
    print(f'  {date}: V = {v:.4f} m³  →  t_sampled ≈ {t_min:.1f} min')

Rod area  = 0.0015 × 0.035 × 2 = 0.000105 m²
D         = 0.1148 m
RPM       = 2700
Formula   : C (grain/m³) = N / V
           V = rod_area × D × π × RPM × t_sampled

Sampling volume V per date:
  2016-04-04: V = 3.0715 m³  →  t_sampled ≈ 30.0 min
  2016-04-05: V = 1.8401 m³  →  t_sampled ≈ 18.0 min
  2016-04-06: V = 1.8402 m³  →  t_sampled ≈ 18.0 min
  2016-04-07: V = 1.8400 m³  →  t_sampled ≈ 18.0 min
  2016-04-08: V = 1.8401 m³  →  t_sampled ≈ 18.0 min
  2016-04-12: V = 3.0702 m³  →  t_sampled ≈ 30.0 min
  2016-04-13: V = 3.0702 m³  →  t_sampled ≈ 30.0 min
  2016-04-14: V = 3.0702 m³  →  t_sampled ≈ 30.0 min
  2016-04-15: V = 3.0704 m³  →  t_sampled ≈ 30.0 min
  2016-04-18: V = 3.0703 m³  →  t_sampled ≈ 30.0 min
  2016-04-19: V = 1.2204 m³  →  t_sampled ≈ 11.9 min
  2016-04-20: V = 3.0702 m³  →  t_sampled ≈ 30.0 min


## Step 3 — Load species-level fungal counts and convert to concentration
Source: `species data.xlsx`  
Each sheet = one sampling date.  
Layout: Row 3 = header (Species | S1 | S2 | S3 | S4 | S5 | S6), then species counts below.

**Conversion:** same formula, same V per date → C_species = N_species / V

In [ ]:
wb2 = openpyxl.load_workbook(SPECIES_DATA)

# Sheet name → date string mapping
date_map = {
    '4th':  '2016-04-04', '5th':  '2016-04-05', '6th':  '2016-04-06',
    '7th':  '2016-04-07', '8th':  '2016-04-08', '12':   '2016-04-12',
    '13':   '2016-04-13', '14':   '2016-04-14', '15':   '2016-04-15',
    '18':   '2016-04-18', '19th': '2016-04-19', '20th': '2016-04-20',
}

# Species to extract (Emericella dropped — 19% missing, max count = 6)
TARGET_SPECIES = [
    'Alternaria sp', 'Aspergillus spores', 'Drechslera sp',
    'Fusarium sp',   'Tetraploa sp.',      'Yeast cell'
]

sp_rows = []
for sheet, date in date_map.items():
    ws2 = wb2[sheet]
    V   = date_V.get(date, np.nan)   # volume for this date
    for r in range(3, ws2.max_row + 1):
        raw = ws2.cell(r, 1).value
        if raw is None:
            continue
        sp = str(raw).strip()
        if sp not in TARGET_SPECIES:
            continue
        for sta in range(1, 7):
            val = ws2.cell(r, sta + 1).value
            # skip None or text values
            n   = None if (val is None or isinstance(val, str)) else float(val)
            # convert count → concentration using same V as total counts
            conc = (n / V) if (n is not None and not np.isnan(V) and V > 0) else np.nan
            sp_rows.append({'date': date, 'station': sta,
                            'species': sp, 'N': n, 'conc': conc})

df_sp = pd.DataFrame(sp_rows)

# Pivot: one row per date-station, each species becomes a column
df_pivot = (
    df_sp.pivot_table(index=['date','station'], columns='species',
                      values='conc', aggfunc='first')
    .reset_index()
)
df_pivot.columns.name = None
df_pivot = df_pivot.rename(columns={
    'Alternaria sp':     'Alternaria_conc',
    'Aspergillus spores':'Aspergillus_conc',
    'Drechslera sp':     'Drechslera_conc',
    'Fusarium sp':       'Fusarium_conc',
    'Tetraploa sp.':     'Tetraploa_conc',
    'Yeast cell':        'Yeast_conc',
})

print(f'Species data shape: {df_pivot.shape}')
print('\nMissing values per species:')
for col in [c for c in df_pivot.columns if '_conc' in c]:
    n = df_pivot[col].isna().sum()
    print(f'  {col:25s}: {n}/{len(df_pivot)} missing')
df_pivot.head()

Species data shape: (67, 8)

Missing values per species:
  Alternaria_conc          : 1/67 missing
  Aspergillus_conc         : 0/67 missing
  Drechslera_conc          : 0/67 missing
  Fusarium_conc            : 7/67 missing
  Tetraploa_conc           : 0/67 missing
  Yeast_conc               : 0/67 missing


,date,station,Alternaria_conc,Aspergillus_conc,Drechslera_conc,Fusarium_conc,Tetraploa_conc,Yeast_conc
0,2016-04-04,1,22.464594,110.369526,0.976721,23.766889,0.651148,9.767215
1,2016-04-04,2,42.324597,171.902979,6.511476,18.883282,2.930164,57.952141
2,2016-04-04,3,13.022953,8.790493,0.000000,0.000000,0.000000,23.766889
3,2016-04-04,4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2016-04-04,5,23.766889,86.928211,0.000000,4.558034,0.000000,2.930164


## Step 4 — Merge datasets and engineer features
### Wind direction circular decomposition
**Problem:** WD is circular — 1° and 359° differ by only 2° but linear models treat them as opposite.  
**Solution:** Decompose into two orthogonal components:
- `sin(WD)` — north–south wind component  
- `cos(WD)` — east–west wind component

This is standard practice in aerobiological modelling (Maya-Manzano et al., 2017).

In [ ]:
# Merge met data with species data on date + station
df = pd.merge(df_met, df_pivot, on=['date','station'], how='left')

# Wind direction circular decomposition
df['sin_WD'] = np.sin(np.deg2rad(df['WD']))
df['cos_WD'] = np.cos(np.deg2rad(df['WD']))

# Fill 2–3 missing species values with column median
for col in ['Aspergillus_conc', 'Alternaria_conc']:
    n_missing = df[col].isna().sum()
    df[col] = df[col].fillna(df[col].median())
    print(f'Filled {n_missing} missing in {col} → median = {df[col].median():.2f}')

# Drop helper columns not needed for analysis
df = df.drop(columns=['N_pollen', 'N_fungus', 'V', 'WD'])

print(f'\nFinal dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')
df[['date','station','pollen_conc','fungus_conc','Aspergillus_conc',
    'Alternaria_conc','GHI','Tamb','RH','WS','sin_WD','cos_WD']].head(8)

Filled 1 missing in Aspergillus_conc → median = 36.41
Filled 2 missing in Alternaria_conc → median = 26.36

Final dataset: 64 rows × 17 columns
Columns: ['date', 'station', 'pollen_conc', 'fungus_conc', 'GHI', 'Tamb', 'RH', 'WS', 'BP', 'Alternaria_conc', 'Aspergillus_conc', 'Drechslera_conc', 'Fusarium_conc', 'Tetraploa_conc', 'Yeast_conc', 'sin_WD', 'cos_WD']


,date,station,pollen_conc,fungus_conc,Aspergillus_conc,Alternaria_conc,GHI,Tamb,RH,WS,sin_WD,cos_WD
0,2016-04-04,1,19.22,215.96,110.369526,22.464594,814,25.0,50,4.7,-1.000000,-1.836970e-16
1,2016-04-04,2,293.16,181.85,171.902979,42.324597,814,25.0,50,4.7,-1.000000,-1.836970e-16
2,2016-04-04,3,7.81,51.79,8.790493,13.022953,814,25.0,50,4.7,-1.000000,-1.836970e-16
3,2016-04-04,5,5.53,121.17,86.928211,23.766889,814,25.0,50,4.7,-1.000000,-1.836970e-16
4,2016-04-05,1,154.34,40.76,83.147655,17.390359,726,28.1,33,1.7,0.829038,-5.591929e-01
5,2016-04-05,2,76.08,123.36,33.693821,32.063475,726,28.1,33,1.7,0.829038,-5.591929e-01
6,2016-04-05,3,12.50,50.54,31.520026,6.521385,726,28.1,33,1.7,0.829038,-5.591929e-01
7,2016-04-05,4,45.65,929.34,714.091625,160.860823,726,28.1,33,1.7,0.829038,-5.591929e-01


## Step 5 — Compute LULC features per station (200 m buffer)
For each sampling station, we compute the percentage area of each LULC class  
within a 200 m radius buffer, matching the proximity analysis in the thesis.

**CRS note:** Shapefiles are in UTM Zone 43N (EPSG:32643). Station GPS coordinates  
(WGS84) are projected to UTM before buffering.

In [ ]:
import os

# Station GPS coordinates (WGS84 lat/lon)
stations_ll = {
    1: (33.645964, 72.980854),   # Entrance gate
    2: (33.638288, 72.989778),   # Open space / lake
    3: (33.643592, 72.997788),   # Near residential area
    4: (33.647927, 72.990664),   # Grid / construction site
    5: (33.645104, 72.988082),   # Rooftop (15.3 m)
    6: (33.644953, 72.988044),   # Indoor window (7.66 m)
}

# Project to UTM for metric-accurate buffering
transformer = Transformer.from_crs('EPSG:4326', 'EPSG:32643', always_xy=True)
station_utm = {s: transformer.transform(lon, lat)
               for s, (lat, lon) in stations_ll.items()}

# Set OGR environment variable to restore missing .shx file
os.environ['SHAPE_RESTORE_SHX'] = 'YES'

# Load LULC shapefiles
vegetation = gpd.read_file(SHP_VEGETATION)
grassland  = gpd.read_file(SHP_GRASSLAND)
barren     = gpd.read_file(SHP_BARREN)
buildings  = gpd.read_file(SHP_BUILDINGS)
tree_road  = gpd.read_file(SHP_TREE_ROAD)
tree_grass = gpd.read_file(SHP_TREE_GRASS)

RADIUS = 200  # metres — matches thesis proximity analysis radius

lulc_rows = []
for sid, (x, y) in station_utm.items():
    buf      = Point(x, y).buffer(RADIUS)
    buf_area = buf.area   # m²

    lulc_rows.append({
        'station':         sid,
        # percentage of 200m buffer covered by each class
        'veg_pct':         round(vegetation.geometry.intersection(buf).area.sum() / buf_area * 100, 1),
        'grass_field_pct': round(grassland.geometry.intersection(buf).area.sum()  / buf_area * 100, 1),
        'barren_pct':      round(barren.geometry.intersection(buf).area.sum()     / buf_area * 100, 1),
        'building_pct':    round(buildings.geometry.intersection(buf).area.sum()  / buf_area * 100, 1),
        'building_count':  int(buildings[buildings.geometry.intersects(buf)].shape[0]),
        'tree_count':      int(
            tree_road[tree_road.geometry.intersects(buf)].shape[0] +
            tree_grass[tree_grass.geometry.intersects(buf)].shape[0]
        ),
    })

df_lulc = pd.DataFrame(lulc_rows)
print('LULC features per station (200 m buffer):')
df_lulc

LULC features per station (200 m buffer):


,station,veg_pct,grass_field_pct,barren_pct,building_pct,building_count,tree_count
0,1,40.6,26.7,0.0,0.2,1,124
1,2,46.2,1.9,25.3,2.3,3,310
2,3,49.6,17.4,0.0,8.2,11,476
3,4,51.4,18.6,0.0,8.3,3,1026
4,5,6.4,54.9,0.8,6.7,7,934
5,6,6.9,55.9,0.7,7.3,8,957


In [ ]:
import os

# Create the 'data' directory if it doesn't exist
os.makedirs('data', exist_ok=True)

# Save both processed datasets
df.to_csv('data/df_analysis.csv', index=False)
df_lulc.to_csv('data/df_lulc.csv', index=False)

print('Saved:')
print(f'  df_analysis.csv  — {df.shape[0]} rows × {df.shape[1]} columns')
print(f'  df_lulc.csv      — {df_lulc.shape[0]} rows × {df_lulc.shape[1]} columns')
print()
print('Column descriptions for df_analysis.csv:')
descriptions = {
    'date':             'Sampling date (YYYY-MM-DD)',
    'station':          'Station number (1–6)',
    'pollen_conc':      'Total pollen concentration (grain/m³)',
    'fungus_conc':      'Total fungal concentration (grain/m³)',
    'GHI':              'Global Horizontal Irradiance (W/m²)',
    'Tamb':             'Ambient temperature (°C)',
    'RH':               'Relative humidity (%)',
    'WS':               'Wind speed (m/s)',
    'BP':               'Barometric pressure (hPa)',
    'sin_WD':           'sin(wind direction) — circular decomposition',
    'cos_WD':           'cos(wind direction) — circular decomposition',
    'Aspergillus_conc': 'Aspergillus concentration (grain/m³)',
    'Alternaria_conc':  'Alternaria concentration (grain/m³)',
    'Drechslera_conc':  'Drechslera concentration (grain/m³)',
    'Fusarium_conc':    'Fusarium concentration (grain/m³)',
    'Tetraploa_conc':   'Tetraploa concentration (grain/m³)',
    'Yeast_conc':       'Yeast cell concentration (grain/m³)',
}
for col, desc in descriptions.items():
    if col in df.columns:
        print(f'  {col:25s} {desc}')

Saved:
  df_analysis.csv  — 64 rows × 17 columns
  df_lulc.csv      — 6 rows × 7 columns

Column descriptions for df_analysis.csv:
  date                      Sampling date (YYYY-MM-DD)
  station                   Station number (1–6)
  pollen_conc               Total pollen concentration (grain/m³)
  fungus_conc               Total fungal concentration (grain/m³)
  GHI                       Global Horizontal Irradiance (W/m²)
  Tamb                      Ambient temperature (°C)
  RH                        Relative humidity (%)
  WS                        Wind speed (m/s)
  BP                        Barometric pressure (hPa)
  sin_WD                    sin(wind direction) — circular decomposition
  cos_WD                    cos(wind direction) — circular decomposition
  Aspergillus_conc          Aspergillus concentration (grain/m³)
  Alternaria_conc           Alternaria concentration (grain/m³)
  Drechslera_conc           Drechslera concentration (grain/m³)
  Fusarium_conc             